In [51]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [52]:
df = pd.read_csv('/content/iot_telemetry_data.csv')
df.head()

,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,1.594512e+09,b8:27:eb:bf:9d:51,0.004956,51.000000,False,0.007651,False,0.020411,22.700000
1,1.594512e+09,00:0f:00:70:91:0a,0.002840,76.000000,False,0.005114,False,0.013275,19.700001
2,1.594512e+09,b8:27:eb:bf:9d:51,0.004976,50.900000,False,0.007673,False,0.020475,22.600000
3,1.594512e+09,1c:bf:ce:15:ec:4d,0.004403,76.800003,True,0.007023,False,0.018628,27.000000
4,1.594512e+09,b8:27:eb:bf:9d:51,0.004967,50.900000,False,0.007664,False,0.020448,22.600000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 405184 entries, 0 to 405183
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   ts        405184 non-null  float64
 1   device    405184 non-null  object 
 2   co        405184 non-null  float64
 3   humidity  405184 non-null  float64
 4   light     405184 non-null  bool   
 5   lpg       405184 non-null  float64
 6   motion    405184 non-null  bool   
 7   smoke     405184 non-null  float64
 8   temp      405184 non-null  float64
dtypes: bool(2), float64(6), object(1)
memory usage: 22.4+ MB


In [4]:
df.describe()

,ts,co,humidity,lpg,smoke,temp
count,4.051840e+05,405184.000000,405184.000000,405184.000000,405184.000000,405184.000000
mean,1.594858e+09,0.004639,60.511694,0.007237,0.019264,22.453987
std,1.994984e+05,0.001250,11.366489,0.001444,0.004086,2.698347
min,1.594512e+09,0.001171,1.100000,0.002693,0.006692,0.000000
25%,1.594686e+09,0.003919,51.000000,0.006456,0.017024,19.900000
50%,1.594858e+09,0.004812,54.900000,0.007489,0.019950,22.200000
75%,1.595031e+09,0.005409,74.300003,0.008150,0.021838,23.600000
max,1.595203e+09,0.014420,99.900002,0.016567,0.046590,30.600000


In [6]:
df.isnull()

,ts,device,co,humidity,light,lpg,motion,smoke,temp
0,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...
405179,False,False,False,False,False,False,False,False,False
405180,False,False,False,False,False,False,False,False,False
405181,False,False,False,False,False,False,False,False,False
405182,False,False,False,False,False,False,False,False,False


***DataPreprocessing*** --> Cleaning raw Datas

In [58]:
df['ts']=pd.to_datetime(df['ts'])
# converting timestamp

In [54]:
df.fillna(method='ffill',inplace=True)
# filling missing values

/tmp/ipykernel_1790/2846587808.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill',inplace=True)


In [55]:
df.drop_duplicates(inplace=True)
# removing duplicates

***Feature Engineering*** --> Here creating new features to improve the model

In [57]:
from sklearn.preprocessing import LabelEncoder
# time -> risk variation by time
df['hour'] = df['ts'].dt.hour
df['day'] = df['ts'].dt.day

# encoding the device
le=LabelEncoder()
df['device'] = le.fit_transform(df['device'])

Creating risk score --> Combining multiple sensor signal

In [21]:
df['risk_score'] = (
    0.3 * df['co'] +
    0.3 * df['lpg']+
    0.2*df['smoke']+
    0.1*df['temp']+
    0.1*df['humidity']
)

Features and targets

In [24]:
X = df.drop(['risk_score','ts'],axis=1)
y = df['risk_score']

Train & Test Split

In [25]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

Training Multiple Models

Linear Regression

In [34]:
lr = LinearRegression()
lr.fit(X_train,y_train)

LinearRegression()

Random Forest

In [43]:
rfr = RandomForestRegressor()
rfr.fit(X_train,y_train)

RandomForestRegressor()

Decision Tress

In [48]:
dtr = DecisionTreeRegressor()
dtr.fit(X_train,y_train)

DecisionTreeRegressor()

Evaluation Function -> measuring model performance using standard metrics

In [60]:
def evaluate_model(name,model):
  pred = model.predict(X_test)

  mean = mean_absolute_error(y_test, pred)
  rmean = np.sqrt(mean_squared_error(y_test,pred))
  r2 = r2_score(y_test,pred)

  print(f"\n{name} Perforance: ")
  print(f"MAE: ",mean)
  print(f"RMSE: ",rmean)
  print("R2 Score: ",r2)

Comparing the Models

In [61]:
evaluate_model("Linear Regression",lr)
evaluate_model("Random Forest",rfr)
evaluate_model("Decission Tree",dtr)


Linear Regression Perforance: 
MAE:  1.2126948218779657e-15
RMSE:  1.613076067726619e-15
R2 Score:  1.0

Random Forest Perforance: 
MAE:  0.00024670008964998484
RMSE:  0.004672023473276759
R2 Score:  0.999980267348192

Decission Tree Perforance: 
MAE:  0.0002442708299779074
RMSE:  0.005748291837312373
R2 Score:  0.9999701287746499
